<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/UPTE_Payscale_TCS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code would take in Job Summary Reports received from UC and convert them into tables for each campus and job titles producing the step tables.

In [ ]:
import pandas as pd

represented_job_codes_df = pd.read_excel('/content/UPTE-26-015_Job Summary Report_3.20.26.xlsx',
                                         sheet_name="Represented Job Codes")
represented_job_codes_df.shape

(121114, 30)

In [ ]:
represented_job_codes_df.columns

Index(['Salary Plan SETID', 'Status - Job Code', 'Job Code',
       'Job Code Description', 'Eff Date - Salary Grade',
       'Salary Grade Eff Status', 'Union Code', 'FLSA Status',
       'Personnel Program/CLASS_INDC', 'OnCall Eligible', 'Elig Shift Diff',
       'Sal Plan', 'Sal Plan Description', 'Grade', 'Rate Code', 'Comp Freq',
       'UCPATH Step', 'UC  Half Step', 'Hrly Rate', 'Monthly Rt', 'Annual Rt',
       'CTO/OS Subgroup', 'CTO Description', 'SOC Code',
       'SOC Code Description', 'Census/OCC Code',
       'Census/OCC Code Description', 'EEO1 Reporting Code',
       'EEO1 Reporting Cod Description', 'OSHPD Code'],
      dtype='object')

In [ ]:
upte_represented_job_codes_df = represented_job_codes_df[represented_job_codes_df['Union Code'].isin(['HX', 'RX', 'TX'])]
upte_represented_job_codes_df.shape


(46098, 30)

In [ ]:
upte_represented_job_codes_df['Job Code'].nunique()

574

In [ ]:
cols_to_keep = ['Salary Plan SETID', 'Job Code', 'Job Code Description', 'Union Code','FLSA Status', 'OnCall Eligible', 'Elig Shift Diff',
                'Rate Code', 'UCPATH Step', 'UC  Half Step', 'Hrly Rate', 'Monthly Rt', 'Annual Rt']
#

Keep only the necessary columns -


1.   'Salary Plan SETID' - Excel file name
2.   'Job Code'
3.   'Job Code Description'    - Second level grouping
1.   'Union Code'.        - Tab name in Excel file
2.   'FLSA Status'        - In Group label for the second level
4.   'OnCall Eligible'.   - In Group label for the second level
5.   'Elig Shift Diff'.   - In Group label for the second level
6.   'Rate Code'.         - In Group label for the second level
7.   'Hrly Rate'          - transpose to column
8.   'Monthly Rt'         - transpose to column
9.   'Annual Rt'          - transpose to column
10.  'Short Descr'.       - First level grouping in tab
11.  'UCPATH Step'        - column label  
12.  'UC  Half Step'      - column label 2  





In [ ]:
upte_represented_job_codes_refined_df = upte_represented_job_codes_df[cols_to_keep]
upte_represented_job_codes_refined_df.columns

Index(['Salary Plan SETID', 'Job Code', 'Job Code Description', 'Union Code',
       'FLSA Status', 'OnCall Eligible', 'Elig Shift Diff', 'Rate Code',
       'UCPATH Step', 'UC  Half Step', 'Hrly Rate', 'Monthly Rt', 'Annual Rt'],
      dtype='object')

In [ ]:
business_unit_list = upte_represented_job_codes_df['Salary Plan SETID'].unique()
business_unit_list

array(['BKCMP', 'DVCMP', 'DVMED', 'IRCMP', 'IRMED', 'LACMP', 'LAMED',
       'LBNL1', 'MECMP', 'RVCMP', 'SBCMP', 'SCCMP', 'SDCMP', 'SDMED',
       'SFCMP', 'SFMED', 'UCANR', 'UCOP1'], dtype=object)

In [ ]:
import math
# dictionary to store the job details
job_details_dict = {}

for business_unit in business_unit_list:
  job_details_dict[business_unit] = {}
  #define the current business unit
  current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

  # loop through the bargaining units
  for bargaining_unit in current_bus_unit['Union Code'].unique():
    job_details_dict[business_unit][bargaining_unit] = {}

    # define the current bargaining unit
    current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

    title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

      # loop through the job_code in the current business unit
    for title in title_list:

        step_list = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['UCPATH Step'].unique()
        half_step_list = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['UC  Half Step'].unique()
        job_details_dict[business_unit][bargaining_unit][title] = {
            'Title' : title,
            'Job Code' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Job Code'].unique(),
            'Exempt Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['FLSA Status'].unique(),
            'OnCall Eligibility Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['OnCall Eligible'].unique(),
            'Shift Differential Status' : current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Elig Shift Diff'].unique(),
            'Steps' : sorted([ int(k) if not math.isnan(k) else 0 for k in step_list]),
            'Half Steps' : sorted([ int(k) if not math.isnan(k) else 0 for k in half_step_list ]),
            'Hourly Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Hrly Rate'].unique()),
            'Monthly Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Monthly Rt'].unique()),
            'Annual Rates': sorted(current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Annual Rt'].unique())
    }



In [ ]:
import pprint

pprint.pprint(job_details_dict['LBNL1']['TX'])

{'Accelerator Oper Principal': {'Annual Rates': [np.float64(nan)],
                                'Exempt Status': array(['N'], dtype=object),
                                'Half Steps': [0],
                                'Hourly Rates': [np.float64(nan)],
                                'Job Code': array(['650.2'], dtype=object),
                                'Monthly Rates': [np.float64(nan)],
                                'OnCall Eligibility Status': array(['N'], dtype=object),
                                'Shift Differential Status': array(['N'], dtype=object),
                                'Steps': [0],
                                'Title': 'Accelerator Oper Principal'},
 'Accelerator Operator': {'Annual Rates': [np.float64(nan)],
                          'Exempt Status': array(['N'], dtype=object),
                          'Half Steps': [0],
                          'Hourly Rates': [np.float64(nan)],
                          'Job Code': array(['650.1'], dtype

create 18 Dataframe structures and write them to EXCEL with separate tabs for each bargaining unit

In [ ]:
# take out UCOP and LBNL to prevent the code from crashing due to not having "HX" in those business units
campus_bu_list = [bus_unit for bus_unit in business_unit_list if bus_unit not in ['UCOP1', 'LBNL1']]

for bus_unit in campus_bu_list:

  barg_units = ['HX', 'RX', 'TX']

  # start the excel writer outside the loop
  with pd.ExcelWriter(f'{bus_unit}_tcs_step_scale_03202026.xlsx', engine='openpyxl') as writer:

    for barg_unit in barg_units:

      campus_bu = job_details_dict[bus_unit][barg_unit]
      # save the job titles in the bargaining unit in a list from the first layer of the dictionary
      curr_bu = pd.DataFrame(campus_bu).T.index.to_list()

      # Create a list to save all the resulting DataFrames in the current campus' bargaining unit
      campus_bu_list = []

    # initiate the excel writer


      for title in curr_bu:
        data = campus_bu[title]
        # Correctly construct header_row as a single-row DataFrame with named columns
        header_row = pd.DataFrame([{
            ' ': 'Job Info',
            'Title': data['Title'],
            'Job Code': data['Job Code'][0],
            'Exempt Status': data['Exempt Status'][0],
            'OnCall Eligibility Status': data['OnCall Eligibility Status'][0],
            'Shift Differential Status': data['Shift Differential Status'][0]
        }])


        # Create half steps row
        half_steps_row = pd.DataFrame([{'Row_Type': 'Half Steps', **{f'Step_{i+1}': half_step for i, half_step in enumerate(data['Half Steps'])}}])
        # Create rate rows
        hourly_rates_row = pd.DataFrame([{'Row_Type': 'Hourly Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Hourly Rates'])}}])
        monthly_rates_row = pd.DataFrame([{'Row_Type': 'Monthly Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Monthly Rates'])}}])
        annual_rates_row = pd.DataFrame([{'Row_Type': 'Annual Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Annual Rates'])}}])

        final_df = pd.concat([header_row,  half_steps_row, hourly_rates_row, monthly_rates_row, annual_rates_row], ignore_index=True)
        trailing_empty_row = pd.DataFrame([[None] * len(final_df.columns)] * 3, columns=final_df.columns)
        final_df = pd.concat([final_df, trailing_empty_row], ignore_index=True)
        campus_bu_list.append(final_df)


        for ind in range(len(campus_bu_list)):
          campus_bu_list[ind].to_excel(writer, sheet_name=barg_unit, startrow=ind*7, index=False)





Exception ignored in: <function ZipFile.__del__ at 0x7996b5207ce0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


In [ ]:
# Now output the UCOP and LBNL with only RX and TX unit
# take out UCOP and LBNL to prevent the code from crashing due to not having "HX" in those business units
remaining_bu_list = ['UCOP1', 'LBNL1']

for bus_unit in remaining_bu_list:

  barg_units = ['RX', 'TX']

  # start the excel writer outside the loop
  with pd.ExcelWriter(f'{bus_unit}_tcs_step_scale_03202026.xlsx', engine='openpyxl') as writer:

    for barg_unit in barg_units:

      campus_bu = job_details_dict[bus_unit][barg_unit]
      # save the job titles in the bargaining unit in a list from the first layer of the dictionary
      curr_bu = pd.DataFrame(campus_bu).T.index.to_list()

      # Create a list to save all the resulting DataFrames in the current campus' bargaining unit
      campus_bu_list = []

    # initiate the excel writer


      for title in curr_bu:
        data = campus_bu[title]
        # Correctly construct header_row as a single-row DataFrame with named columns
        header_row = pd.DataFrame([{
            ' ': 'Job Info',
            'Title': data['Title'],
            'Job Code': data['Job Code'][0],
            'Exempt Status': data['Exempt Status'][0],
            'OnCall Eligibility Status': data['OnCall Eligibility Status'][0],
            'Shift Differential Status': data['Shift Differential Status'][0]
        }])


        # Create half steps row
        half_steps_row = pd.DataFrame([{'Row_Type': 'Half Steps', **{f'Step_{i+1}': half_step for i, half_step in enumerate(data['Half Steps'])}}])
        # Create rate rows
        hourly_rates_row = pd.DataFrame([{'Row_Type': 'Hourly Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Hourly Rates'])}}])
        monthly_rates_row = pd.DataFrame([{'Row_Type': 'Monthly Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Monthly Rates'])}}])
        annual_rates_row = pd.DataFrame([{'Row_Type': 'Annual Rates', **{f'Step_{i+1}': f'${rate:,.2f}' for i, rate in enumerate(data['Annual Rates'])}}])

        final_df = pd.concat([header_row,  half_steps_row, hourly_rates_row, monthly_rates_row, annual_rates_row], ignore_index=True)
        trailing_empty_row = pd.DataFrame([[None] * len(final_df.columns)] * 3, columns=final_df.columns)
        final_df = pd.concat([final_df, trailing_empty_row], ignore_index=True)
        campus_bu_list.append(final_df)


        for ind in range(len(campus_bu_list)):
          campus_bu_list[ind].to_excel(writer, sheet_name=barg_unit, startrow=ind*7, index=False)





In [ ]:
# Give explicit column names to the other DataFrames for better clarity
# steps_row = pd.DataFrame.from_dict({'UCPATH Step': title_dict[title]['Steps']}, orient='index')
# half_steps_row = pd.DataFrame.from_dict({'UC Half Step': title_dict[title]['Half Steps']}, orient='index')
# hourly_rates_row = pd.DataFrame.from_dict({'Hrly Rate': title_dict[title]['Hourly Rates']}, orient='index')
# monthly_rates_row = pd.DataFrame.from_dict({'Monthly Rt': title_dict[title]['Monthly Rates']}, orient='index')
# annual_rates_row = pd.DataFrame.from_dict({'Annual Rt': title_dict[title]['Annual Rates']}, orient='index')